In [1]:
import os
import requests
from pathlib import Path
from bs4 import BeautifulSoup

In [2]:
from typing import Dict

In [3]:
BASE_URL = "https://licindia.in"
BASE_FOLDER = "lic_policies"

In [4]:
url = f"{BASE_URL}/web/guest/insurance-plan"

In [5]:
def getContent(url: str) -> bytes|None:
    try:
        response = requests.get(url)
        if response.status_code != 200:
            raise Exception
        return response.content
    except Exception as e:
        print("Something is wrong")
        return None

In [6]:
def getSoup(url: str) -> BeautifulSoup:
    try:
        content = getContent(url)
        if content:
            soup = BeautifulSoup(content, 'html.parser')
            return soup
    except Exception as e:
        print(e)

In [7]:
def getPagesForInsurancePlans(soup: BeautifulSoup) -> Dict:
    elements = [elem.find('a').find_next('ul', class_='child-menu') for elem in soup.select('li.has-sub')]
    elems = set(elem.find_next('ul', class_="child-menu") for elem in elements if "Insurance" in elem.find_next('li').find_next('a')['title']\
         and "Plan" in elem.find_next('li').find_next('a')['title'])
    
    policy_map = {}

    for elem in elems:
        links = elem.find_all('a')
        for link in links:
            policy_map[link['title'].strip().replace(' ', '_')] = link['href']
            
    return policy_map

In [8]:
def getInsurancePlanLinks(policyPage: Dict) -> Dict:
    policy_links = {}
    for p, l in policyPage.items():
        soup = getSoup(l)
        policy_links[p] = {link.text.strip().replace("LIC's ", '').replace("LIC’s ", '').replace(' ', '_').replace('(', '') \
                           .replace(')', '').replace(':', ''): link['href'] \
                           for link in soup.find_all('a') if 'LIC' in link.text and 'https' not in link['href'] and 'guest' in link['href']}
    return policy_links

In [9]:
def getPolicyLinksPerPlan(base: str, planLinks: Dict) -> Dict:
    pdfLinks = {}
    for plan, links in planLinks.items():
        policyMap = {}
        for policy, link in links.items():
            soup = getSoup(f"{base}/{link}")
            pdf = [link['href'] for link in soup.find_all('a') if "Policy Document" in link.text][0]
            pdf = pdf[: pdf.index('pdf') + 3]
            policyMap[policy] = pdf
        pdfLinks[plan] = policyMap
    return pdfLinks

In [10]:
def downloadPolicyDocuments(url: str, pdfs: Dict, folder: str):
    for plan, policyPdfs in pdfs.items():
        for policy, pdf in policyPdfs.items():
            file_location = Path(f"{folder}/{plan}")
            if not file_location.is_dir():
                file_location.mkdir(parents=True, exist_ok=True)
            with open(f"{file_location}/{policy}.pdf", 'wb') as f:
                pdf_link = f"{url}/{pdf}"
                f.write(getContent(pdf_link))

In [11]:
def run(url: str, location: str):
    soup = getSoup(url)
    plan_pages = getPagesForInsurancePlans(soup)
    plan_links = getInsurancePlanLinks(plan_pages)
    # return plan_links
    planPdfs = getPolicyLinksPerPlan(url, plan_links)
    downloadPolicyDocuments(url, planPdfs, location)
    return "OK"

In [12]:
run(BASE_URL, BASE_FOLDER)

'OK'